# Feature Engineering

- Transform preprocessed data from `cookie`, `index`, `method`, `url`, `host`, and `label`, `payload` into numerical or categorical features that can be used for modeling.
- Focus on features that distinguish anomalous requests (anom) from normal requests (norm).
- Address missing values in contentLength (e.g., fill with 0 or add a binary flag).
- Convert categorical columns `method`, `label`, `host` into a format suitable for machine learning (e.g., one-hot encoding, label encoding).
- Extract features that capture HTTP request characteristics (e.g., SQL injection patterns, payload length, URL length).
- Save the transformed DataFrame with features as `csic_2010_featurised.csv` for modeling.

In [1]:
## 1. Load the Preprocessed Data

In [22]:
import os
import re
import random
from pathlib import Path

import pandas as pd
from sklearn.preprocessing import OneHotEncoder

path = Path.cwd()
data_path = path.parent / "data"
input_path = data_path / "csic_2010_preprocessed.csv" 
output_path = data_path / "csic_2010_featurised.csv" 
# Load the DataFrame with keep_default_na=False to preserve empty strings
#df = pd.read_csv(input_path, keep_default_na=False)

In [23]:
# Load the Parquet file
%time
df = pd.read_parquet(input_path.with_suffix('.parquet'))

CPU times: user 9 μs, sys: 2 μs, total: 11 μs
Wall time: 27.2 μs


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61065 entries, 0 to 61064
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   cookie         61065 non-null  object 
 1   index          61065 non-null  int64  
 2   method         61065 non-null  object 
 3   url            61065 non-null  object 
 4   host           61065 non-null  object 
 5   label          61065 non-null  object 
 6   payload        61065 non-null  object 
 7   contentLength  17977 non-null  float64
dtypes: float64(1), int64(1), object(6)
memory usage: 3.7+ MB


## 2. Handle Missing Values

- Handle missing `contentLength` - fill with `0`
- Handle Empty `payload` - add binary flag to indicate whether re has payload

In [25]:
df['contentLength'] = df['contentLength'].fillna(0)

## 3. Analyse column values to derive features

In [26]:
print(f"cookie:        {df['cookie'].nunique()}")
print(f"index:         {df['index'].nunique()}")
print(f"method:        {df['method'].nunique()}")
print(f"url:           {df['url'].nunique()}")
print(f"host:          {df['host'].nunique()}")
print(f"label:         {df['label'].nunique()}")
print(f"payload:       {df['payload'].nunique()}")
print(f"contentLength: {df['contentLength'].nunique()}")

cookie:        61065
index:         36000
method:        3
url:           1643
host:          2
label:         2
payload:       12093
contentLength: 383


`index` column is non-unique and reused across requests so it is not useful in modeling. Drop the column.

In [27]:
df_featurised = df.drop(columns = 'index')

### 3.1 Analyse `cookie` column values

In [28]:
# Random cookie samples
random_row_numbers = random.sample(range(0, 61066), 5)
for rn in random_row_numbers:
    print(f"row#{rn}: {df_featurised.iloc[rn]['cookie']}")

row#51998: JSESSIONID=D96668784C61E669E6085763770D0D90
row#12152: JSESSIONID=327BD6AF403AC2AA9A6B3E59A65566CE
row#43903: JSESSIONID=B7ADC8A1E61A94D5E63D1B45583EC40E
row#29397: JSESSIONID=7A453C282A097420BD940657BE4F9DD1
row#1797: JSESSIONID=0772314744A6095FD0B709A279D74A33


In [29]:
# Compute the length of each cookie
cookie_lengths = df_featurised['cookie'].str.len()

print(f"Min len: {cookie_lengths.min()}")
print(f"Max len: {cookie_lengths.max()}")
print(f"mean len: {cookie_lengths.mean():.2f}")
#print(f"Unique lengths: {cookie_lengths.nunique()}")

# Count how many cookies match the pattern
cookie_pattern=r'^JSESSIONID=[A-Z0-9]+$'
valid_count = df_featurised['cookie'].str.match(cookie_pattern).sum()
print(f"Number of valid cookies: {valid_count}")
#print(f"Total cookies: {len(df_featurised)}")

Min len: 43
Max len: 43
mean len: 43.00
Number of valid cookies: 61065


All cookies have a length of `43` characters and a consistent format. This implies that the cookie column does not provide any variability that could distinguish between normal (norm) and anomalous (anom) requests. Drop the `cookie` Column from the Feature set

In [30]:
df_featurised = df.drop(columns = 'cookie')

### 3.2 Analyse `Method` column values

In [31]:
df_featurised['method'].value_counts()

method
GET     43088
POST    17580
PUT       397
Name: count, dtype: int64

Indicates type of HTTP requests - majority are `GET` `43,088` followed by `POST` `17,580` and the `PUT` `397` which can influence anomaly detection  
e.g., `POST` and `PUT` requests are more likely to contain payloads or be anomalous  

Use one-hot encoding to transform method categories into numerical features.

In [32]:
# Initialize the encoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
category_col = ["method"]

# Fit the encoder to the data
encoder.fit(df_featurised[category_col])
print(f"Encoder categories:\n\t{encoder.categories_}")

# Generate column names for the encoded features
encoded_cols = list(encoder.get_feature_names_out(category_col))
print(f"Generated column names:\n\t{encoded_cols}")

Encoder categories:
	[array(['GET', 'POST', 'PUT'], dtype=object)]
Generated column names:
	['method_GET', 'method_POST', 'method_PUT']


The method column is transformed into three binary features - `method_GET`, `method_POST`, `method_PUT`. This allows the model to interpret the request type as numerical input while preserving the categorical relationships.

Apply the encoder to the dataset and Concatenate the encoded features with the original DataFrame for modeling.

In [33]:
df_featurised[encoded_cols] = encoder.transform(df_featurised[category_col]).copy()

In [34]:
# Drop the original 'method' column as it is longer needed
df_featurised.drop(columns=["method"], inplace=True)

In [35]:
df_featurised.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61065 entries, 0 to 61064
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   index          61065 non-null  int64  
 1   url            61065 non-null  object 
 2   host           61065 non-null  object 
 3   label          61065 non-null  object 
 4   payload        61065 non-null  object 
 5   contentLength  61065 non-null  float64
 6   method_GET     61065 non-null  float64
 7   method_POST    61065 non-null  float64
 8   method_PUT     61065 non-null  float64
dtypes: float64(4), int64(1), object(4)
memory usage: 4.2+ MB


### 3.3 Analyse `host` column values

In [36]:
df_featurised['host'].value_counts()

host
localhost:8080    60668
localhost:9090      397
Name: count, dtype: int64

`host` is what type of feature category column ? give justificaiton for documentation

- Identifies the `server:port` for the request.
- Contains two host values `localhost:8080` and `localhost:9090`  
- Majority are `localhost:8080` `60,668` following by `localhost:9090` `397`
- Useful for filtering or grouping requests by host.
- `host` column is highly imbalanced, which may limit its predictive power.

In [37]:
# Initialize the encoder
host_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
category_col = ["host"]

# Fit the encoder to the data
host_encoder.fit(df_featurised[category_col])
print(f"Encoder categories:\n\t{host_encoder.categories_}")

# Generate column names for the encoded features
host_encoded_cols = list(host_encoder.get_feature_names_out(category_col))
print(f"Generated column names:\n\t{host_encoded_cols}")

Encoder categories:
	[array(['localhost:8080', 'localhost:9090'], dtype=object)]
Generated column names:
	['host_localhost:8080', 'host_localhost:9090']


The host column is transformed into two binary features - method_GET, method_POST, method_PUT. This allows the model to interpret the request type as numerical input while preserving the categorical relationships.

Apply the encoder to the dataset and concatenate the encoded features with the original DataFrame for modeling.

In [38]:
df_featurised[host_encoded_cols] = host_encoder.transform(df_featurised[category_col]).copy()
df_featurised.head(5)

,index,url,host,label,payload,contentLength,method_GET,method_POST,method_PUT,host_localhost:8080,host_localhost:9090
0,6883,http://localhost:8080/tienda1/imagenes/2.gif,localhost:8080,norm,,0.0,1.0,0.0,0.0,1.0,0.0
1,4687,http://localhost:8080/tienda1/publico/vaciar.j...,localhost:8080,anom,,0.0,1.0,0.0,0.0,1.0,0.0
2,25623,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,norm,id=2,0.0,1.0,0.0,0.0,1.0,0.0
3,10367,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,anom,id=3 nombre=Jam%F3n+Ib%E9rico precio=%2500 can...,0.0,1.0,0.0,0.0,1.0,0.0
4,19522,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,anom,id=2sessionid%3D12312312%26+username%3D%3Cscri...,174.0,0.0,1.0,0.0,1.0,0.0


### 3.4 Analyse URL column values

Count unique URLs in the dataset

In [39]:
print(f"Unique URLs: {df_featurised['url'].nunique()}")

Unique URLs: 1643


Sample a few random URLs for analysis

In [40]:
random_row_numbers = random.sample(range(0, 61066), 10)
for rn in random_row_numbers:
    print(f"row[{rn}]: {df_featurised.iloc[rn]['url']}")

row[24559]: http://localhost:8080/tienda1/publico/entrar.jsp
row[41625]: http://localhost:8080/tienda1/index.jsp
row[13973]: http://localhost:8080/tienda1/global/titulo.jsp
row[3980]: http://localhost:8080/tienda1/publico/entrar.jsp
row[38603]: http://localhost:8080/tienda1/publico/autenticar.jsp
row[59502]: http://localhost:8080/tienda1/publico/autenticar.jsp
row[8367]: http://localhost:8080/theme
row[50708]: http://localhost:8080/tienda1/publico/productos.jsp
row[5348]: http://localhost:8080/tienda1/publico/carrito.jsp
row[15752]: http://localhost:8080/tienda1/publico/pagar.jsp


**Observations:**  
URLs contain patterns, structures, and metadata that can be leveraged for feature engineering.  
Eg:
- url length
- Count of special chars: `/` `.` `?` `&` etc
- Presence of query prarams
- Presence of suspecious patterns: sql select, union, drop etc
- Presence of path traversal

Analyse `urls` in detail for presence of `special chars`, `patterns` etc that are `label`'ed  anomalous

In [41]:
from collections import defaultdict

#url_df = df_featurised["url"].head(10).to_frame()
#url_df = df_featurised["url", "label"].to_frame()
url_df = df_featurised[["url", "label"]]

# Special characters
special_chars = ['/', '.', '?', '&', '=', '%']
total_urls_per_special_char = defaultdict(int)

# SQL patterns
sql_patterns = [ "select", "union", "drop", "insert", "delete"]
total_urls_per_sql_pattern = defaultdict(int)
for pattern in sql_patterns:
    total_urls_per_sql_pattern[pattern] = 0

# Script Pattern
script_regex_pattern = r"<script|%3cscript"
total_urls_with_script_pattern = 0

# Alert Pattern
alert_regex_pattern = r"alert|%61%6c%65%72%74"
total_urls_with_alert_pattern = 0

# OR pattern
or_regex_pattern = r"\bor\b|\|\|"
total_urls_with_or_pattern = 0

# Comment pattern
comment_regex_pattern = r"--|/\*|\*/|%2d%2d"
total_urls_with_comment_pattern = 0

# Path traversal pattern
path_traversal_regex_pattern = r"\.\./|%2e%2e%2f"
total_urls_with_path_traversal_pattern = 0

# Parameter pattern
parameter_regex_pattern = r"[?&][^=&]+="
total_urls_with_parameter_pattern = 0

# Trailing digits
trailing_digits_regex_pattern= r"http://localhost:8080[^0-9]*(\d+)"
total_urls_with_trailing_digits_pattern = 0

# Numeric ID
numeric_id_regex_pattern = r"/\d{10,}"
total_urls_with_numeric_id_pattern = 0

# Track label distribution for each pattern
label_distribution = {
    "select": {"anom": 0, "norm": 0},
    "drop": {"anom": 0, "norm": 0},
    "script": {"anom": 0, "norm": 0},
    "alert": {"anom": 0, "norm": 0},
    "or": {"anom": 0, "norm": 0},
    "comment": {"anom": 0, "norm": 0},
    "path_traversal": {"anom": 0, "norm": 0},
    "parameter": {"anom": 0, "norm": 0},
    "trailing_digits": {"anom": 0, "norm": 0},
    "numeric_id": {"anom": 0, "norm": 0},
}

# Iterate through each URL and its corresponding label
for url, label in zip(url_df["url"], url_df["label"]):

    # Count special characters
    for char in special_chars:
        count = url.count(char)
        if count > 0:
            total_urls_per_special_char[char] += 1
        else:
            total_urls_per_special_char[char] += 0

    for pattern in sql_patterns:
        if pattern in url:
          total_urls_per_sql_pattern[pattern] +=  1
          label_distribution[pattern][label] += 1
        
    if re.search(script_regex_pattern, url):
        total_urls_with_script_pattern += 1
        label_distribution["script"][label] += 1
        
    if re.search(alert_regex_pattern, url):
        total_urls_with_alert_pattern += 1
        label_distribution["alert"][label] += 1

    if re.search(or_regex_pattern, url):
        total_urls_with_or_pattern += 1
        label_distribution["or"][label] += 1
        
    if re.search(comment_regex_pattern, url):
        total_urls_with_comment_pattern += 1
        label_distribution["comment"][label] += 1
        
    if re.search(path_traversal_regex_pattern, url):
        total_urls_with_path_traversal_pattern += 1
        label_distribution["path_traversal"][label] += 1

    if re.search(parameter_regex_pattern, url):
        total_urls_with_parameter_pattern += 1
        label_distribution["parameter"][label] += 1
    
    if re.search(trailing_digits_regex_pattern, url):
        total_urls_with_trailing_digits_pattern += 1
        label_distribution["trailing_digits"][label] += 1
        
    if re.search(numeric_id_regex_pattern, url):
        total_urls_with_numeric_id_pattern += 1
        label_distribution["numeric_id"][label] += 1
    
# Display the total count of URLs containing each special character
print("Total URLs containing per special Character:")
for char, count in total_urls_per_special_char.items():
    print(f"    {char}: {count} URLs")

print("Total URLs containing per SQL pattern:")
for pattern, count in total_urls_per_sql_pattern.items():
    print(f"    {pattern}: {count} URLs")

print(f"Total URLs containing other patterns:")
print(f"   'or': {total_urls_with_or_pattern}")
print(f"   'script': {total_urls_with_script_pattern}")
print(f"   'alert': {total_urls_with_alert_pattern}")
print(f"   'comment': {total_urls_with_comment_pattern}")
print(f"   'path traversal': {total_urls_with_path_traversal_pattern}")
print(f"   'parameter': {total_urls_with_parameter_pattern}")
print(f"   'trailing digits': {total_urls_with_trailing_digits_pattern}")
print(f"   'numeric id': {total_urls_with_numeric_id_pattern}")

# Display label distribution for each pattern
print("\nLabel distribution for patterns:")
for pattern, counts in label_distribution.items():
    print(f"   {pattern}: {counts}")


Total URLs containing per special Character:
    /: 61065 URLs
    .: 60386 URLs
    ?: 0 URLs
    &: 0 URLs
    =: 0 URLs
    %: 0 URLs
Total URLs containing per SQL pattern:
    select: 8 URLs
    union: 0 URLs
    drop: 5 URLs
    insert: 0 URLs
    delete: 0 URLs
Total URLs containing other patterns:
   'or': 0
   'script': 0
   'alert': 0
   'comment': 0
   'path traversal': 0
   'parameter': 0
   'trailing digits': 59329
   'numeric id': 967

Label distribution for patterns:
   select: {'anom': 8, 'norm': 0}
   drop: {'anom': 5, 'norm': 0}
   script: {'anom': 0, 'norm': 0}
   alert: {'anom': 0, 'norm': 0}
   or: {'anom': 0, 'norm': 0}
   comment: {'anom': 0, 'norm': 0}
   path_traversal: {'anom': 0, 'norm': 0}
   parameter: {'anom': 0, 'norm': 0}
   trailing_digits: {'anom': 23329, 'norm': 36000}
   numeric_id: {'anom': 967, 'norm': 0}


**Observations:**

- Special Characters:  
  - `/` and `.` are too common and do not help distinguish anomalies.  
  - `?`, `&`, `=`, `%` do not appear in the dataset, so they provide no signal.  
  > **Conclusion:** Exclude all special character counts from the model

- SQL Patterns:  
  - `select` and `drop` are rare but meaningful for detecting SQL injection.  
  - `union`, `insert`, `delete` do not appear in the dataset.
  > **Conclusion:** Include `select` and `drop` as binary features to for detecting SQL injection attempts.

- Other Patterns:  
  - All patterns `or`, `script`, `alert`, `comment`, `path traversal`, `parameter` have zero occurrences in the dataset.  
  - `Trailing digits` appears in both anom and norm URLs and is not discriminative
  - `Numeric ID` is a strong indicator of anomalies. All of its occurrences are in anom URLs.
  > **Conclusion:** Include `numeric_id` as a binary feature. Exclude all other patterns from the model.

Add the `url` derived feature columns to dataframe

In [42]:
df_featurised["url_has_select"] = df_featurised["url"].str.contains("select", case=False).astype(int)
df_featurised["url_has_drop"] = df_featurised["url"].str.contains("drop", case=False).astype(int)
df_featurised["url_has_numeric_id"] = df_featurised["url"].apply(lambda x: 1 if re.search(r"/\d{10,}", x) else 0).astype(int)

# Additional features
df_featurised["url_length"] = df_featurised["url"].apply(len)

# Display the new features
#df_featurised[["url", "label", "url_has_select", "url_has_drop", "url_has_numeric_id", "url_length"]].head()
df_featurised.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61065 entries, 0 to 61064
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   index                61065 non-null  int64  
 1   url                  61065 non-null  object 
 2   host                 61065 non-null  object 
 3   label                61065 non-null  object 
 4   payload              61065 non-null  object 
 5   contentLength        61065 non-null  float64
 6   method_GET           61065 non-null  float64
 7   method_POST          61065 non-null  float64
 8   method_PUT           61065 non-null  float64
 9   host_localhost:8080  61065 non-null  float64
 10  host_localhost:9090  61065 non-null  float64
 11  url_has_select       61065 non-null  int64  
 12  url_has_drop         61065 non-null  int64  
 13  url_has_numeric_id   61065 non-null  int64  
 14  url_length           61065 non-null  int64  
dtypes: float64(6), int64(5), object(4)
m

### 3.5 Analyse `payload` column values

Count unique `payloads` in the dataset

In [43]:
print(f"Unique Payloads: {df_featurised['payload'].nunique()}")

Unique Payloads: 12093


Sample a few random Payloads for analysis

In [44]:
unique_values = df_featurised['payload'].unique()
random_row_numbers = random.sample(range(0, len(unique_values)), 10)
for rn in random_row_numbers:
    print(f"    {unique_values[rn]}\n")

    id=1 nombreA=Jam%F3n+Ib%E9rico precio=85 cantidad=85 B1=A%F1adir+al+carrito

    modo=insertar precio=3064 B1=Confirmar

    modoA=insertar precio=4307 B1=Confirmar

    modo=registro login=katzenel password=+ nombre=Yenina apellidos=Cat%E0+Viera email=mabuchi-britts%40portaldesign.tj dni=72130723D direccion=C%2F+Los+Sauces%2C+87+ ciudad=Albag%E9s%2C+L%27 cp=31220 provincia=Segovia ntc=8743026490183484 B1=Registrar

    modo=registro login=ramiz password=garantir nombre=Enrico apellidos=Vallb%F3+Olmos email=pangborn%40informex.me dni=07485924E direccion=C%2F+Las+Chumberas%2C+38%2C+ ciudad=Medina+de+Rioseco cp=40320 provincia=Segovia ntc=9431159156347522 B1=Registrar

    modo=registro login=jeanloui password=any%253F%250D%250ASet-cookie%253A%2BTamper%253D1041264011025374727 nombre=Evelio apellidos=Fonfria+Castillero email=sprerris-apfel%40tresbrujas.bn dni=89061921X direccion=C%2F+Cervantes+117%2C+10%3FG ciudad=%DArbel+del+Castillo cp=50343 provincia=Palencia ntc=9895940721818753 B

**Observations:**  
- Payloads show form submissions, login attempts, data entries etc 
- Typically formatted as `key-value` pairs (e.g., modo=registro, login=quek, password=5a69io99t3).
- Feature can be derived based on structure, encoded chars, sql keywords, long numeric string, suspicious patterns etc

Analyse payloads in detail for presence of special chars, patterns etc that are label'ed anomalous

In [45]:
from collections import defaultdict

payload_df = df_featurised[["payload", "label"]]

# Special characters
special_chars = ['&', '=', '%']
total_payloads_per_special_char = defaultdict(int)

# Count digits
digits_regex_pattern= r"\d"
total_payloads_with_digits_pattern = 0

# Parameter pattern
#parameter_regex_pattern = r"[?&][^=&]+="
parameter_regex_pattern = r"(^|&)[^=&]+="
total_payloads_with_parameter_pattern = 0

# Quote pattern
quote_regex_pattern = r"'|%27|\"|%22"
total_payloads_with_quote_pattern = 0

# Angle bracket pattern
angle_bkt_regex_pattern = r"<|>|%3c|%3e"
total_payloads_with_angle_bkt_pattern = 0

# Paranthesis pattern
paranthesis_regex_pattern = r"\(|\)|%28|%29"
total_payloads_with_paranthesis_pattern = 0

# Semicolon pattern
semicolon_regex_pattern = r";|%3b"
total_payloads_with_semicolon_pattern = 0

# Slash pattern
slash_regex_pattern = r"/|%2f"
total_payloads_with_slash_pattern = 0

# Comment pattern
comment_regex_pattern = r"--|/\*|\*/|%2d%2d"
total_payloads_with_comment_pattern = 0

# Percent encoded pattern
percent_encoded_regex_pattern = r"%[0-9a-fA-F]{2}"
total_payloads_with_percent_encoded_pattern = 0

# SQL patterns
sql_patterns = [ "select", "union", "drop", "insert", "delete"]
total_payloads_per_sql_pattern = defaultdict(int)
for pattern in sql_patterns:
    total_payloads_per_sql_pattern[pattern] = 0

# Script Pattern
script_regex_pattern = r"<script|%3cscript"
total_payloads_with_script_pattern = 0

# Alert Pattern
alert_regex_pattern = r"alert|%61%6c%65%72%74"
total_payloads_with_alert_pattern = 0

# OR pattern
or_regex_pattern = r"\bor\b|\|\|"
total_payloads_with_or_pattern = 0

# Path traversal pattern
path_traversal_regex_pattern = r"\.\./|%2e%2e%2f"
total_payloads_with_path_traversal_pattern = 0


# Track label distribution for each pattern
payload_label_distribution = {
    "digits": {"anom": 0, "norm": 0},
    "parameter": {"anom": 0, "norm": 0},    
    "quote": {"anom": 0, "norm": 0},
    "angle_bkt": {"anom": 0, "norm": 0},
    "paranthesis": {"anom": 0, "norm": 0},
    "semicolon": {"anom": 0, "norm": 0},
    "slash": {"anom": 0, "norm": 0},
    "comment": {"anom": 0, "norm": 0},
    "percent_encoded": {"anom": 0, "norm": 0},
    
    "select": {"anom": 0, "norm": 0},
    "union": {"anom": 0, "norm": 0},
    "drop": {"anom": 0, "norm": 0},
    "insert": {"anom": 0, "norm": 0},
    "delete": {"anom": 0, "norm": 0},
 
    "script": {"anom": 0, "norm": 0},
    "alert": {"anom": 0, "norm": 0},
    "or": {"anom": 0, "norm": 0},
    "path_traversal": {"anom": 0, "norm": 0},    
}

# Iterate through each Payload and its corresponding label
for payload, label in zip(payload_df["payload"], payload_df["label"]):

    # Count special characters
    for char in special_chars:
        count = payload.count(char)
        if count > 0:
            total_payloads_per_special_char[char] += 1
        else:
            total_payloads_per_special_char[char] += 0

    if re.search(digits_regex_pattern, payload):
        total_payloads_with_digits_pattern += 1
        payload_label_distribution["digits"][label] += 1

    if re.search(parameter_regex_pattern, payload):
        total_payloads_with_parameter_pattern += 1
        payload_label_distribution["parameter"][label] += 1

    if re.search(quote_regex_pattern, payload):
        total_payloads_with_quote_pattern += 1
        payload_label_distribution["quote"][label] += 1
    
    if re.search(angle_bkt_regex_pattern, payload):
        total_payloads_with_angle_bkt_pattern += 1
        payload_label_distribution["angle_bkt"][label] += 1

    if re.search(paranthesis_regex_pattern, payload):
        total_payloads_with_paranthesis_pattern += 1
        payload_label_distribution["paranthesis"][label] += 1

    if re.search(semicolon_regex_pattern, payload):
        total_payloads_with_semicolon_pattern += 1
        payload_label_distribution["semicolon"][label] += 1
    
    if re.search(slash_regex_pattern, payload):
        total_payloads_with_slash_pattern += 1
        payload_label_distribution["slash"][label] += 1    

    if re.search(percent_encoded_regex_pattern, payload):
        total_payloads_with_percent_encoded_pattern += 1
        payload_label_distribution["percent_encoded"][label] += 1

    for pattern in sql_patterns:
        if pattern in payload:
          total_payloads_per_sql_pattern[pattern] +=  1
          payload_label_distribution[pattern][label] += 1
        
    if re.search(script_regex_pattern, payload):
        total_payloads_with_script_pattern += 1
        payload_label_distribution["script"][label] += 1
        
    if re.search(alert_regex_pattern, payload):
        total_payloads_with_alert_pattern += 1
        payload_label_distribution["alert"][label] += 1

    if re.search(or_regex_pattern, payload):
        total_payloads_with_or_pattern += 1
        payload_label_distribution["or"][label] += 1
        

    if re.search(path_traversal_regex_pattern, payload):
        total_payloads_with_path_traversal_pattern += 1
        payload_label_distribution["path_traversal"][label] += 1

# Display the total count of payloads containing each special character
print("Total payloads containing per special Character:")
for char, count in total_payloads_per_special_char.items():
    print(f"    {char}: {count} payloads")

print("Total payloads containing per SQL pattern:")
for pattern, count in total_payloads_per_sql_pattern.items():
    print(f"    {pattern}: {count} payloads")

print(f"Total payloads containing other patterns:")
print(f"   'parameter': {total_payloads_with_parameter_pattern}")
print(f"   'digits': {total_payloads_with_digits_pattern}")
print(f"   'quote': {total_payloads_with_quote_pattern}")
print(f"   'angle_bkt': {total_payloads_with_angle_bkt_pattern}")
print(f"   'paranthesis': {total_payloads_with_paranthesis_pattern}")
print(f"   'semicolon': {total_payloads_with_semicolon_pattern}")
print(f"   'slash': {total_payloads_with_slash_pattern}")
print(f"   'comment': {total_payloads_with_comment_pattern}")
print(f"   'percent_encoded': {total_payloads_with_percent_encoded_pattern}")
print(f"   'script': {total_payloads_with_script_pattern}")
print(f"   'alert': {total_payloads_with_alert_pattern}")
print(f"   'or': {total_payloads_with_or_pattern}")
print(f"   'path traversal': {total_payloads_with_path_traversal_pattern}")


# Display label distribution for each pattern
print("\nLabel distribution for patterns:")
for pattern, counts in payload_label_distribution.items():
    print(f"   {pattern}: {counts}")


Total payloads containing per special Character:
    &: 0 payloads
    =: 35574 payloads
    %: 21395 payloads
Total payloads containing per SQL pattern:
    select: 0 payloads
    union: 0 payloads
    drop: 4 payloads
    insert: 4588 payloads
    delete: 0 payloads
Total payloads containing other patterns:
   'parameter': 35574
   'digits': 32629
   'quote': 4188
   'angle_bkt': 0
   'paranthesis': 1554
   'semicolon': 0
   'slash': 17
   'comment': 0
   'percent_encoded': 21395
   'script': 0
   'alert': 1158
   'or': 2
   'path traversal': 0

Label distribution for patterns:
   digits: {'anom': 18629, 'norm': 14000}
   parameter: {'anom': 19574, 'norm': 16000}
   quote: {'anom': 4088, 'norm': 100}
   angle_bkt: {'anom': 0, 'norm': 0}
   paranthesis: {'anom': 1524, 'norm': 30}
   semicolon: {'anom': 0, 'norm': 0}
   slash: {'anom': 17, 'norm': 0}
   comment: {'anom': 0, 'norm': 0}
   percent_encoded: {'anom': 15001, 'norm': 6394}
   select: {'anom': 0, 'norm': 0}
   union: {'anom':

**Observations:** 
- Special Characters:
  - `&` no occurances and `=` is too common. Both these are not useful
  - `%` is also common but may be useful when combined with `percent_encoded` pattern
  > **Conclusion:**  Exclude `&` and `=`. Use `%` as a feature

- SQL Patterns:  
  -  No occurances `select` `union`, `delete`
  -  `insert` appears in both `anom` and `norm` but is more common in `anom` (56% of occurrences are anomalous).
  - `drop` appears only in `norm`, so it is not useful for anomaly detection.  
  > **Conclusion:** Include `insert` as binary feature. Exclude all other SQL patterns.  

- Other Patterns:
  - `>90%` of `quote` `paranthesis` `slash` `alert` are anomalous. Thsee are strong candidates features  
  - `>50%` of `Parameter` `percent_eccoded` `insert` are anomalous. Provides resonable discriminative value
  - Rest all are give almost no signals for anomalous classification 
  > **Conclusion:** Invlude `quote` `paranthesis` `slash` `alert` `Parameter` `percent_eccoded` `insert` drop the rest

Add the `payload` derived feature columns in the dataset

In [46]:
# Extract the `Payload` columns series
payload_series = df_featurised["payload"].astype(str)

# Count based features
df_featurised["payload_quote_count"] = payload_series.str.count(r"'|%27|\"|%22", flags=re.IGNORECASE)
df_featurised["payload_paren_count"] = payload_series.str.count(r"\(|\)|%28|%29", flags=re.IGNORECASE)
df_featurised["payload_slash_count"] = payload_series.str.count(r"/|%2f", flags=re.IGNORECASE)
df_featurised["payload_parameter_count"] = payload_series.str.count(r"(^|&)[^=&]+=", flags=re.IGNORECASE)
df_featurised["payload_percent_count"] = payload_series.str.count(r"%", flags=re.IGNORECASE)
df_featurised["payload_percent_encoded_count"] = payload_series.str.count(r"%[0-9a-fA-F]{2}", flags=re.IGNORECASE)

# Binary flag based features
df_featurised["payload_has_alert"] = payload_series.str.contains(r"alert|%61%6c%65%72%74", case=False, regex=True)
df_featurised["payload_has_sql_insert"] = payload_series.str.contains(r"insert", case=False, regex=True)

# Additional features
df_featurised["payload_length"] = payload_series.str.len()

df_featurised.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61065 entries, 0 to 61064
Data columns (total 24 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   index                          61065 non-null  int64  
 1   url                            61065 non-null  object 
 2   host                           61065 non-null  object 
 3   label                          61065 non-null  object 
 4   payload                        61065 non-null  object 
 5   contentLength                  61065 non-null  float64
 6   method_GET                     61065 non-null  float64
 7   method_POST                    61065 non-null  float64
 8   method_PUT                     61065 non-null  float64
 9   host_localhost:8080            61065 non-null  float64
 10  host_localhost:9090            61065 non-null  float64
 11  url_has_select                 61065 non-null  int64  
 12  url_has_drop                   61065 non-null 

In [47]:
df_featurised.head(5)

,index,url,host,label,payload,contentLength,method_GET,method_POST,method_PUT,host_localhost:8080,...,url_length,payload_quote_count,payload_paren_count,payload_slash_count,payload_parameter_count,payload_percent_count,payload_percent_encoded_count,payload_has_alert,payload_has_sql_insert,payload_length
0,6883,http://localhost:8080/tienda1/imagenes/2.gif,localhost:8080,norm,,0.0,1.0,0.0,0.0,1.0,...,44,0,0,0,0,0,0,False,False,0
1,4687,http://localhost:8080/tienda1/publico/vaciar.j...,localhost:8080,anom,,0.0,1.0,0.0,0.0,1.0,...,70,0,0,0,0,0,0,False,False,0
2,25623,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,norm,id=2,0.0,1.0,0.0,0.0,1.0,...,57,0,0,0,1,0,0,False,False,4
3,10367,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,anom,id=3 nombre=Jam%F3n+Ib%E9rico precio=%2500 can...,0.0,1.0,0.0,0.0,1.0,...,48,0,0,0,1,4,4,False,False,77
4,19522,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,anom,id=2sessionid%3D12312312%26+username%3D%3Cscri...,174.0,0.0,1.0,0.0,1.0,...,57,2,0,5,1,18,18,False,False,174


In [48]:
### 3.6 Analyse `contentLength` column values

In [49]:
print(f"Unique values: {df_featurised['contentLength'].nunique()}")
df_featurised['contentLength'].describe()

Unique values: 383


count    61065.000000
mean        31.936903
std         75.660779
min          0.000000
25%          0.000000
50%          0.000000
75%         17.000000
max        836.000000
Name: contentLength, dtype: float64

In [62]:
# Assuming content_length_df contains 'contentLength' and 'label' columns
content_length_df = df_featurised[["contentLength", "label"]]

In [67]:
# Initialize a dictionary to store label distribution for each statistical range
content_length_label_distribution = {
    "0_to_10": {"anom": 0, "norm": 0},
    "10_to_50": {"anom": 0, "norm": 0},
    "50_to_200": {"anom": 0, "norm": 0},
    "greater_than_200": {"anom": 0, "norm": 0},
}
for contentLength, label in zip(content_length_df["contentLength"], content_length_df["label"]):
    # Categorize contentLength into bins based on the statistics
    if contentLength > 0 and contentLength <= 100:
        content_length_label_distribution["0_to_10"][label] += 1
    elif contentLength > 100 and contentLength <= 300:
        content_length_label_distribution["10_to_50"][label] += 1
    elif contentLength > 300 and contentLength <= 500:
        content_length_label_distribution["50_to_200"][label] += 1
    else: # >200
        content_length_label_distribution["greater_than_200"][label] += 1

# Display the label distribution for each statistical range
print("Label Distribution for contentLength Ranges:")
for range_key, label_distr in content_length_label_distribution.items():
    print(f"   {range_key}: {label_distr}")


Label Distribution for contentLength Ranges:
   0_to_10: {'anom': 6281, 'norm': 6000}
   10_to_50: {'anom': 3207, 'norm': 1996}
   50_to_200: {'anom': 409, 'norm': 4}
   greater_than_200: {'anom': 15168, 'norm': 28000}


In [50]:
import numpy as np
from collections import defaultdict

# Assuming content_length_df contains 'contentLength' and 'label' columns
content_length_df = df_featurised[["contentLength", "label"]]

# Define the statistical thresholds
stats = {
    "min": 0.0,
    "25%": 0.0,
    "50%": 0.0,
    "75%": 17.0,
    "max": 836.0,
    "mean": 31.936903,
    "std": 75.660779
}

# Initialize a dictionary to store label distribution for each statistical range
label_distribution = {
    "min_to_25%": {"anom": 0, "norm": 0},
    "25%_to_50%": {"anom": 0, "norm": 0},
    "50%_to_75%": {"anom": 0, "norm": 0},
    "75%_to_mean": {"anom": 0, "norm": 0},
    "mean_to_mean+std": {"anom": 0, "norm": 0},
    "mean+std_to_max": {"anom": 0, "norm": 0},
    "zero": {"anom": 0, "norm": 0}
}

# Iterate through each contentLength and its corresponding label
for contentLength, label in zip(content_length_df["contentLength"], content_length_df["label"]):
    # Categorize contentLength into bins based on the statistics
    if contentLength == 0:
        label_distribution["zero"][label] += 1
    elif 0 < contentLength <= stats["25%"]:
        label_distribution["min_to_25%"][label] += 1
    elif stats["25%"] < contentLength <= stats["50%"]:
        label_distribution["25%_to_50%"][label] += 1
    elif stats["50%"] < contentLength <= stats["75%"]:
        label_distribution["50%_to_75%"][label] += 1
    elif stats["75%"] < contentLength <= stats["mean"]:
        label_distribution["75%_to_mean"][label] += 1
    elif stats["mean"] < contentLength <= stats["mean"] + stats["std"]:
        label_distribution["mean_to_mean+std"][label] += 1
    elif contentLength > stats["mean"] + stats["std"]:
        label_distribution["mean+std_to_max"][label] += 1

# Display the label distribution for each statistical range
print("Label Distribution for contentLength Ranges:")
for range_name, counts in label_distribution.items():
    total = counts["anom"] + counts["norm"]
    anom_percent = (counts["anom"] / total * 100) if total > 0 else 0
    norm_percent = (counts["norm"] / total * 100) if total > 0 else 0
    print(f"   {range_name}: {counts} (anom: {anom_percent:.2f}%, norm: {norm_percent:.2f}%)")

Label Distribution for contentLength Ranges:
   min_to_25%: {'anom': 0, 'norm': 0} (anom: 0.00%, norm: 0.00%)
   25%_to_50%: {'anom': 0, 'norm': 0} (anom: 0.00%, norm: 0.00%)
   50%_to_75%: {'anom': 808, 'norm': 2000} (anom: 28.77%, norm: 71.23%)
   75%_to_mean: {'anom': 638, 'norm': 0} (anom: 100.00%, norm: 0.00%)
   mean_to_mean+std: {'anom': 4946, 'norm': 4000} (anom: 55.29%, norm: 44.71%)
   mean+std_to_max: {'anom': 3585, 'norm': 2000} (anom: 64.19%, norm: 35.81%)
   zero: {'anom': 15088, 'norm': 28000} (anom: 35.02%, norm: 64.98%)


In [53]:
# Assuming df_featurised contains 'contentLength' and 'label' columns
content_length_df = df_featurised[["contentLength", "label"]]

# Group by 'contentLength' and 'label', then count occurrences
length_label_counts = content_length_df.groupby(["contentLength", "label"]).size().unstack(fill_value=0)

# Add a column for total frequency of each contentLength
length_label_counts["total"] = length_label_counts.sum(axis=1)

# Sort by contentLength for readability
length_label_counts = length_label_counts.sort_index()

# Calculate the percentage of 'anom' and 'norm' for each contentLength
length_label_counts["anom_percent"] = (length_label_counts["anom"] / length_label_counts["total"] * 100).round(2)
length_label_counts["norm_percent"] = (length_label_counts["norm"] / length_label_counts["total"] * 100).round(2)

# Display the results
print("Frequency and Label Distribution for Each contentLength:")
print(length_label_counts)


Frequency and Label Distribution for Each contentLength:
label           anom   norm  total  anom_percent  norm_percent
contentLength                                                 
0.0            15088  28000  43088         35.02         64.98
4.0               57   1000   1057          5.39         94.61
5.0              458      0    458        100.00          0.00
6.0               39      0     39        100.00          0.00
7.0               18      0     18        100.00          0.00
...              ...    ...    ...           ...           ...
822.0              1      0      1        100.00          0.00
824.0              2      0      2        100.00          0.00
826.0              1      0      1        100.00          0.00
835.0              1      0      1        100.00          0.00
836.0              1      0      1        100.00          0.00

[383 rows x 5 columns]


In [58]:
length_label_counts[(length_label_counts["anom_percent"] == 100.00)].sort_index()

label,anom,norm,total,anom_percent,norm_percent
contentLength,,,,,
5.0,458,0,458,100.0,0.0
6.0,39,0,39,100.0,0.0
7.0,18,0,18,100.0,0.0
8.0,74,0,74,100.0,0.0
9.0,37,0,37,100.0,0.0
...,...,...,...,...,...
822.0,1,0,1,100.0,0.0
824.0,2,0,2,100.0,0.0
826.0,1,0,1,100.0,0.0


In [55]:
length_label_counts[(length_label_counts["norm_percent"] == 100.00)].sort_index()

label,anom,norm,total,anom_percent,norm_percent
contentLength,,,,,


### 3.6 Analyse `label` column values

In [ ]:
df_featurised['label'].value_counts()

In [ ]:
`label` is what type of feature category column ? give justificaiton for documentation`label` is what type of feature category column ? give justificaiton for documentation

### Analyse `contentLength` column values

In [ ]:
print(f"Unique values: {df_featurised['contentLength'].nunique()}")
df_featurised['contentLength'].describe()

`contentLength` is what type of feature category column ? give justificaiton for documentation

In [ ]:
# Add binary flag to indicate whether a request has a payload:
df["payload_is_empty"] = (payload_series == "").astype(int)

In [ ]:
df.info()

## 3. Encode Categorical Variables

Convert categorical columns `method`, `label`, `host` into a format suitable for machine learning (e.g., one-hot encoding, label encoding)

## 4. Create Domain-Specific Features  

Extract features that capture HTTP request characteristics (e.g., SQL injection patterns, payload length, URL length).


## 5. Save the Feature-Engineered DataFrame to disk

Save the final DataFrame (with features) as `csic_2010_engineered.csv` for modeling.